In [1]:
import os
data_dir = '/home/devang/Projects/PilotCrew/Data-Science-Bench/tasks/task_13/data'
print(os.listdir(data_dir))

['reservations.csv', 'tables.csv', 'orders.csv']


In [2]:
import pandas as pd
reservations = pd.read_csv(os.path.join(data_dir, 'reservations.csv'))
tables = pd.read_csv(os.path.join(data_dir, 'tables.csv'))
orders = pd.read_csv(os.path.join(data_dir, 'orders.csv'))

print("Reservations columns:", reservations.columns)
print("Tables columns:", tables.columns)
print("Orders columns:", orders.columns)

Reservations columns: Index(['reservation_id', 'reservation_time', 'party_size', 'table_id',
       'channel', 'no_show', 'seated_time'],
      dtype='str')
Tables columns: Index(['table_id', 'section', 'capacity'], dtype='str')
Orders columns: Index(['order_id', 'reservation_id', 'server', 'food_total', 'drink_total',
       'duration_min'],
      dtype='str')


In [3]:
print(reservations.head())
print(orders.head())

  reservation_id     reservation_time  party_size table_id channel  no_show  \
0          R0001  2024-11-01 19:00:00           3     TB03  walkin        0   
1          R0002  2024-11-01 21:00:00           4     TB04  online        0   
2          R0003  2024-11-01 23:00:00           5     TB05   phone        0   
3          R0004  2024-11-02 19:00:00           4     TB04  online        0   
4          R0005  2024-11-02 21:00:00           6     TB05   phone        0   

           seated_time  
0  2024-11-01 19:04:00  
1  2024-11-01 21:07:00  
2  2024-11-01 23:10:00  
3  2024-11-02 19:05:00  
4  2024-11-02 21:08:00  
  order_id reservation_id server  food_total  drink_total  duration_min
0    O0001          R0001    Ana          53           20            67
1    O0002          R0002     Bo          62           26            69
2    O0003          R0003   Chen          71           32            71
3    O0004          R0004   Chen          63           28            70
4    O0005     

In [4]:
# Question: Among walk-ins that were seated on Friday through Sunday at 19:00 or later, 
# compute revenue per occupied seat-minute, where revenue is food_total + drink_total 
# and occupied seat-minutes are party_size * duration_min. 
# Which server has the highest median revenue per occupied seat-minute? 
# If there is a tie, use alphabetical order.

# 1. Filter reservations:
# - walk-ins (channel == 'walkin')
# - seated (no_show == 0 or seated_time is not null)
# - Friday through Sunday (dayofweek >= 4)
# - 19:00 or later (hour >= 19)

reservations['seated_time'] = pd.to_datetime(reservations['seated_time'])
reservations['reservation_time'] = pd.to_datetime(reservations['reservation_time'])

# Let's check if seated_time or reservation_time should be used for "at 19:00 or later" and "Friday through Sunday"
# The prompt says "seated on Friday through Sunday at 19:00 or later". This implies seated_time.
# Let's use seated_time.

walkins = reservations[(reservations['channel'] == 'walkin') & (reservations['no_show'] == 0)].copy()
walkins['day_of_week'] = walkins['seated_time'].dt.dayofweek # Monday=0, Sunday=6
walkins['hour'] = walkins['seated_time'].dt.hour

filtered_walkins = walkins[(walkins['day_of_week'] >= 4) & (walkins['hour'] >= 19)]

# Merge with orders
merged = pd.merge(filtered_walkins, orders, on='reservation_id')

# Compute revenue and occupied seat-minutes
merged['revenue'] = merged['food_total'] + merged['drink_total']
merged['occupied_seat_minutes'] = merged['party_size'] * merged['duration_min']
merged['revenue_per_osm'] = merged['revenue'] / merged['occupied_seat_minutes']

# Group by server and compute median
server_stats = merged.groupby('server')['revenue_per_osm'].median().reset_index()
server_stats = server_stats.sort_values(by=['revenue_per_osm', 'server'], ascending=[False, True])

print(server_stats)


  server  revenue_per_osm
3    Dia         0.382114
0    Ana         0.312114
2   Chen         0.306099
1     Bo         0.296913


In [5]:
# Double check the filtering
print("Total reservations:", len(reservations))
walkins = reservations[(reservations['channel'] == 'walkin') & (reservations['seated_time'].notnull())].copy()
print("Total seated walkins:", len(walkins))

walkins['day_of_week'] = walkins['seated_time'].dt.dayofweek
walkins['time'] = walkins['seated_time'].dt.time

import datetime
filtered_walkins = walkins[(walkins['day_of_week'] >= 4) & (walkins['time'] >= datetime.time(19, 0))]
print("Filtered walkins:", len(filtered_walkins))

merged = pd.merge(filtered_walkins, orders, on='reservation_id')
print("Merged with orders:", len(merged))

merged['revenue'] = merged['food_total'] + merged['drink_total']
merged['occupied_seat_minutes'] = merged['party_size'] * merged['duration_min']
merged['revenue_per_osm'] = merged['revenue'] / merged['occupied_seat_minutes']

server_stats = merged.groupby('server')['revenue_per_osm'].median().reset_index()
server_stats = server_stats.sort_values(by=['revenue_per_osm', 'server'], ascending=[False, True])
print(server_stats)


Total reservations: 84
Total seated walkins: 25
Filtered walkins: 9
Merged with orders: 9
  server  revenue_per_osm
3    Dia         0.382114
0    Ana         0.312114
2   Chen         0.306099
1     Bo         0.296913


In [6]:
print(merged[['reservation_id', 'server', 'revenue_per_osm']])

  reservation_id server  revenue_per_osm
0          R0001    Ana         0.363184
1          R0006     Bo         0.372549
2          R0008   Chen         0.381643
3          R0044     Bo         0.481013
4          R0046    Dia         0.382114
5          R0051    Ana         0.261044
6          R0064   Chen         0.230556
7          R0069     Bo         0.203509
8          R0071     Bo         0.221277


In [7]:
# Let's check if there are any other walkins that were seated on Friday through Sunday at 19:00 or later
# Maybe the seated_time is not the only indicator of being seated.
# The prompt says "seated on Friday through Sunday at 19:00 or later".
# Let's check if there are any walkins with no_show == 0 but seated_time is null.
print(reservations[(reservations['channel'] == 'walkin') & (reservations['no_show'] == 0) & (reservations['seated_time'].isnull())])

Empty DataFrame
Columns: [reservation_id, reservation_time, party_size, table_id, channel, no_show, seated_time]
Index: []


In [8]:
# Let's check if "at 19:00 or later" means reservation_time or seated_time.
# "Among walk-ins that were seated on Friday through Sunday at 19:00 or later"
# Usually walk-ins don't have a reservation time, or their reservation time is the same as their arrival time.
print(reservations[reservations['channel'] == 'walkin'][['reservation_time', 'seated_time']].head())

      reservation_time         seated_time
0  2024-11-01 19:00:00 2024-11-01 19:04:00
5  2024-11-02 23:00:00 2024-11-02 23:11:00
7  2024-11-03 21:00:00 2024-11-03 21:09:00
9  2024-11-04 19:00:00 2024-11-04 19:07:00
14 2024-11-05 23:00:00 2024-11-05 23:14:00


In [ ]:
# Let's check if we use reservation_time instead of seated_time for the "at 19:00 or later" condition
walkins = reservations[(reservations['channel'] == 'walkin') & (reservations['seated_time'].notnull())].copy()
walkins['day_of_week'] = walkins['seated_time'].dt.dayofweek
walkins['res_time'] = walkins['reservation_time'].dt.time

filtered_walkins_res = walkins[(walkins['day_of_week'] >= 4) & (walkins['res_time'] >= datetime.time(19, 0))]
print("Filtered walkins (res time):", len(filtered_walkins_res))

merged_res = pd.merge(filtered_walkins_res, orders, on='reservation_id')
merged_res['revenue'] = merged_res['food_total'] + merged_res['drink_total']
merged_res['occupied_seat_minutes'] = merged_res['party_size'] * merged_res['duration_min']
merged_res['revenue_per_osm'] = merged_res['revenue'] / merged_res['occupied_seat_minutes']

server_stats_res = merged_res.groupby('server')['revenue_per_osm'].median().reset_index()
server_stats_res = server_stats_res.sort_values(by=['revenue_per_osm', 'server'], ascending=[False, True])
print(server_stats_res)